##NAME:UNIBHA NEUPANE GROUP:L5CG11

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        # Stopping conditions
        if len(unique_classes) == 1:
            return {"class": unique_classes[0]}
        if num_samples == 0 or (self.max_depth is not None and depth >= self.max_depth):
            return {"class": np.bincount(y).argmax()}

        best_gain = -1
        best_split = None

        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold

                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue

                gain = self._information_gain(y, y[left_mask], y[right_mask])

                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        "feature_idx": feature_idx,
                        "threshold": threshold,
                        "left_mask": left_mask,
                        "right_mask": right_mask
                    }

        if best_split is None:
            return {"class": np.bincount(y).argmax()}

        left_tree = self._build_tree(
            X[best_split["left_mask"]],
            y[best_split["left_mask"]],
            depth + 1
        )

        right_tree = self._build_tree(
            X[best_split["right_mask"]],
            y[best_split["right_mask"]],
            depth + 1
        )

        return {
            "feature_idx": best_split["feature_idx"],
            "threshold": best_split["threshold"],
            "left_tree": left_tree,
            "right_tree": right_tree
        }

    def _entropy(self, y):
        probabilities = np.bincount(y) / len(y)
        return -np.sum(probabilities * np.log2(probabilities + 1e-9))

    def _information_gain(self, parent, left, right):
        return self._entropy(parent) - (
            (len(left) / len(parent)) * self._entropy(left)
            + (len(right) / len(parent)) * self._entropy(right)
        )

    def predict(self, X):
        return [self._predict_single(x, self.tree) for x in X]

    def _predict_single(self, x, tree):
        if "class" in tree:
            return tree["class"]
        if x[tree["feature_idx"]] <= tree["threshold"]:
            return self._predict_single(x, tree["left_tree"])
        else:
            return self._predict_single(x, tree["right_tree"])


In [ ]:
data = load_iris()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)

y_pred_sklearn = sklearn_tree.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("Scikit-learn Decision Tree Accuracy:", accuracy_sklearn)


Scikit-learn Decision Tree Accuracy: 1.0


In [ ]:
print("Accuracy Comparison")

# Instantiate, train, and evaluate the Custom Decision Tree
custom_tree = CustomDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)
y_pred_custom = custom_tree.predict(X_test)
accuracy_custom = accuracy_score(y_test, y_pred_custom)

print("Custom Decision Tree:", accuracy_custom)
print("Scikit-learn Decision Tree:", accuracy_sklearn)

Accuracy Comparison
Custom Decision Tree: 1.0
Scikit-learn Decision Tree: 1.0


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
rf_pred = rf.predict(X_test)

print("Decision Tree F1 Score:", f1_score(y_test, dt_pred, average="weighted"))
print("Random Forest F1 Score:", f1_score(y_test, rf_pred, average="weighted"))


Decision Tree F1 Score: 0.9439974457215836
Random Forest F1 Score: 1.0


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1_weighted"
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)


Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best F1 Score: 0.9782952128219708


In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

dt_reg = DecisionTreeRegressor(random_state=42)
rf_reg = RandomForestRegressor(random_state=42)

dt_reg.fit(X_train, y_train)
rf_reg.fit(X_train, y_train)

dt_pred = dt_reg.predict(X_test)
rf_pred = rf_reg.predict(X_test)

print("Decision Tree MSE:", mean_squared_error(y_test, dt_pred))
print("Random Forest MSE:", mean_squared_error(y_test, rf_pred))


Decision Tree MSE: 0.495235205629094
Random Forest MSE: 0.2553684927247781


In [34]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist,
    n_iter=5,
    cv=3,
    scoring="neg_mean_squared_error"
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)


Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': None}
